# ISIC 2024 - Data Preprocessing Pipeline

This notebook builds a preprocessing pipeline for the ISIC 2024 dataset.

## Objectives
- Clean and prepare tabular data
- Handle missing values
- Encode categorical features
- Select usable features (avoid leakage)
- Prepare data for model training

## Key Considerations
- Severe class imbalance
- High missing values in some features
- Train/Test feature mismatch
- Patient-level data already handled in splitting

In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv("../data/splits/train_split.csv", low_memory=False)
val_df = pd.read_csv("../data/splits/val_split.csv", low_memory=False)
test_df = pd.read_csv("../data/splits/test_split.csv", low_memory=False)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (320848, 55)
Val: (40105, 55)
Test: (40106, 55)


In [3]:
train_ids = train_df["isic_id"].copy()
val_ids = val_df["isic_id"].copy()
test_ids = test_df["isic_id"].copy()

### Feature Selection (Initial)

Removed:
- Diagnostic columns (`iddx_*`, `mel_*`) -> high missing / leakage risk
- Model-derived feature (`tbp_lv_dnn_lesion_confidence`) -> leakage

Only features available at inference time are kept.

In [4]:
DROP_COLUMNS = [
    # target leakage / not in test
    "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
    "mel_mitotic_index", "mel_thick_mm",
    "tbp_lv_dnn_lesion_confidence",

    # identifiers (not useful for model)
    "patient_id", "lesion_id"
]

train_df = train_df.drop(columns=[col for col in DROP_COLUMNS if col in train_df.columns])
val_df = val_df.drop(columns=[col for col in DROP_COLUMNS if col in val_df.columns])
test_df = test_df.drop(columns=[col for col in DROP_COLUMNS if col in test_df.columns])

In [5]:
y_train = train_df["target"]
y_val = val_df["target"]

train_df = train_df.drop(columns=["target"])
val_df = val_df.drop(columns=["target"])

In [6]:
train_df = train_df.drop(columns=["isic_id"])
val_df = val_df.drop(columns=["isic_id"])
test_df = test_df.drop(columns=["isic_id"])

### Feature Types

- Categorical features will be encoded
- Numerical features will be imputed and scaled later

In [7]:
categorical_cols = train_df.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = train_df.select_dtypes(exclude=["object"]).columns.tolist()

C:\Users\phang\AppData\Local\Temp\ipykernel_21248\2278989782.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_df.select_dtypes(include=["object"]).columns.tolist()


### Missing Value Strategy

- Numerical → filled with **median** (robust to outliers)
- Categorical → filled with `"unknown"`

This ensures no information leakage from validation/test sets.

In [8]:
# Numerical
for col in numerical_cols:
    median = train_df[col].median()
    train_df[col] = train_df[col].fillna(median)
    val_df[col] = val_df[col].fillna(median)
    test_df[col] = test_df[col].fillna(median)

# Categorical
for col in categorical_cols:
    train_df[col] = train_df[col].fillna("unknown")
    val_df[col] = val_df[col].fillna("unknown")
    test_df[col] = test_df[col].fillna("unknown")

### Encoding Strategy

- Label Encoding is used for simplicity
- Encoders are fitted on training data only
- Same mapping is applied to validation and test sets

In [9]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    
    train_df[col] = le.fit_transform(train_df[col])
    
    # handle unseen
    val_df[col] = val_df[col].map(lambda x: x if x in le.classes_ else "unknown")
    test_df[col] = test_df[col].map(lambda x: x if x in le.classes_ else "unknown")
    
    le.classes_ = np.append(le.classes_, "unknown")
    
    val_df[col] = le.transform(val_df[col])
    test_df[col] = le.transform(test_df[col])
    
    encoders[col] = le

### Scaling

- StandardScaler is applied to numerical features
- Fit only on training data to prevent leakage

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
val_df[numerical_cols] = scaler.transform(val_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

In [11]:
train_processed = train_df.copy()
val_processed = val_df.copy()
test_processed = test_df.copy()

train_processed["target"] = y_train
val_processed["target"] = y_val

train_processed["isic_id"] = train_ids
val_processed["isic_id"] = val_ids
test_processed["isic_id"] = test_ids

In [12]:
IMAGE_DIR = "../data/raw/ISIC_2024_Training_Input/"

train_processed["image_path"] = IMAGE_DIR + train_processed["isic_id"] + ".jpg"
val_processed["image_path"] = IMAGE_DIR + val_processed["isic_id"] + ".jpg"
test_processed["image_path"] = IMAGE_DIR + test_processed["isic_id"] + ".jpg"

In [13]:
import os

print("Missing images:",
      (~train_processed["image_path"].apply(os.path.exists)).sum())

Missing images: 0


In [14]:
train_processed.to_csv("../data/processed/train_processed.csv", index=False)
val_processed.to_csv("../data/processed/val_processed.csv", index=False)
test_processed.to_csv("../data/processed/test_processed.csv", index=False)

print("Preprocessed data saved!")

Preprocessed data saved!


## Summary

- Removed unusable and leakage-prone features
- Handled missing values
- Encoded categorical variables
- Scaled numerical features

## Ready for:
- PyTorch Dataset / DataLoader
- Model training